# 🟡 L06 — Optional Extensions

> *Pure self-study material. Skipping this will not affect any later lesson.*

Four sections:

1. **Stationarity + differencing** — why ARIMA needs it, how to test for it, how to fix non-stationary series
2. **ACF and PACF** — the diagnostic plots that guide ARIMA parameter choice
3. **ARIMA + AutoARIMA** — the classical model behind every textbook
4. **Prophet** — Facebook's "decompose then forecast" library, popular and easy

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (13, 4.5)

# Same dataset as the Core notebooks
df = pd.read_csv("data/northstar_daily_revenue.csv", parse_dates=["date"]).set_index("date")
y = df["revenue_gbp"]
TEST_SIZE = 60
y_train = y.iloc[:-TEST_SIZE]
y_test  = y.iloc[-TEST_SIZE:]
print("✅ Setup complete.")

# 1. Stationarity and Differencing

**Stationarity** = the statistical properties (mean, variance) don't change over time.

ARIMA models assume stationarity. Most real-world data is NOT stationary (NorthStar revenue has an upward trend). The fix: **differencing** — replace each value with the difference from the previous one.

`y_diff = y - y.shift(1)` removes a linear trend. `y_diff_2 = y_diff - y_diff.shift(1)` removes a quadratic trend (rarely needed).

### Test for stationarity — Augmented Dickey-Fuller

`adfuller()` returns a p-value. Low p (< 0.05) → series is stationary.

In [ ]:
def adf_report(s, name):
    s = s.dropna()
    result = adfuller(s)
    print(f"{name}:  ADF statistic = {result[0]:.3f},  p-value = {result[1]:.4f}")
    if result[1] < 0.05:
        print(f"   → STATIONARY (p < 0.05)")
    else:
        print(f"   → NON-STATIONARY (need to difference)")
    print()

adf_report(y_train, "Original revenue series")

y_diff = y_train.diff()
adf_report(y_diff, "First difference (Δy)")

y_diff_2 = y_diff.diff()
adf_report(y_diff_2, "Second difference (Δ²y) — usually overkill")

**Interpretation:** the original series is NON-stationary (p > 0.05). After one differencing, it becomes stationary. That tells us ARIMA's `d` parameter (number of differences) = 1.

# 2. ACF and PACF

The **autocorrelation function (ACF)** shows how a series correlates with its own lagged versions. Spikes at lag k mean lag-k values predict the current value.

The **partial autocorrelation function (PACF)** is similar but with intermediate lags' effects removed.

Together they tell you ARIMA's `p` (AR order) and `q` (MA order):
- PACF: significant spikes at lags 1..p, then cuts off → AR(p)
- ACF: significant spikes at lags 1..q, then cuts off → MA(q)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(y_train.dropna(), lags=30, ax=axes[0,0], title="ACF of original series")
plot_pacf(y_train.dropna(), lags=30, ax=axes[0,1], title="PACF of original series")
plot_acf(y_diff.dropna(), lags=30, ax=axes[1,0], title="ACF of first difference")
plot_pacf(y_diff.dropna(), lags=30, ax=axes[1,1], title="PACF of first difference")

plt.tight_layout()
plt.show()

print("Look at the BOTTOM ROW (after differencing):")
print("  - ACF spikes at lag 7, 14, 21 → weekly seasonality (need SARIMA, not plain ARIMA)")
print("  - PACF spike at lag 1 → suggests AR(1) on top")

# 3. ARIMA + AutoARIMA

`ARIMA(p, d, q)`:
- `p` = autoregressive order (use the past p values)
- `d` = differencing order (we found d=1)
- `q` = moving-average order (use the past q errors)

For non-seasonal data, ARIMA is great. For seasonal data (like NorthStar), use SARIMA (next section).

### Manual ARIMA

In [ ]:
# ARIMA(1, 1, 1) — moderate AR, one difference, moderate MA
manual = ARIMA(y_train, order=(1, 1, 1)).fit()
print("Manual ARIMA(1, 1, 1):")
print(f"  AIC: {manual.aic:.1f}")

forecast_manual = manual.forecast(steps=TEST_SIZE)
mae_manual = mean_absolute_error(y_test, forecast_manual)
print(f"  Test MAE: £{mae_manual:,.0f}")

### AutoARIMA (manual grid search version)

The `pmdarima` library has a proper AutoARIMA. We don't want an extra dependency for one demo. Below is a small DIY search.

In [ ]:
# Small grid search — try p in {0,1,2}, q in {0,1,2}, d fixed to 1
best_aic = np.inf
best_order = None
best_model = None

for p in [0, 1, 2]:
    for q in [0, 1, 2]:
        try:
            m = ARIMA(y_train, order=(p, 1, q)).fit()
            if m.aic < best_aic:
                best_aic = m.aic
                best_order = (p, 1, q)
                best_model = m
        except Exception:
            continue

print(f"Best ARIMA order by AIC: {best_order} (AIC = {best_aic:.1f})")
forecast_auto = best_model.forecast(steps=TEST_SIZE)
mae_auto = mean_absolute_error(y_test, forecast_auto)
print(f"Test MAE: £{mae_auto:,.0f}")

# 4. SARIMA — Seasonal ARIMA

For seasonal data, SARIMA adds `(P, D, Q, s)` — seasonal versions of (p, d, q) — where `s` is the seasonal period.

For NorthStar (weekly seasonality): `seasonal_order=(P, D, Q, 7)` with similar small values.

In [ ]:
# SARIMA(1, 1, 1) × (1, 1, 1, 7)
sarima = SARIMAX(
    y_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

print(f"SARIMA(1,1,1)(1,1,1,7): AIC = {sarima.aic:.1f}")
forecast_sarima = sarima.forecast(steps=TEST_SIZE)
mae_sarima = mean_absolute_error(y_test, forecast_sarima)
print(f"Test MAE: £{mae_sarima:,.0f}")
print()
print("For comparison from the Core notebooks:")
print(f"  Naive:        MAE £1,270")
print(f"  ETS-weekly:   MAE £2,047")
print(f"  ML-recursive: MAE £2,552")

### 💡 Why SARIMA didn't dominate

- **SARIMA, like ETS-weekly, captures ONE seasonality.** Our data has BOTH weekly and annual. The annual holiday spike is still missed.
- SARIMA is a reasonable choice when data has ONE strong seasonality and limited training history.
- For multi-seasonal data, **ML-with-lags or Prophet** remain better choices.

# 5. Prophet (Facebook's library)

Prophet is "decomposition then forecast" — it fits a flexible additive model: trend + weekly seasonality + yearly seasonality + holidays. Designed for analysts; defaults usually work.

In [ ]:
try:
    from prophet import Prophet
    HAS_PROPHET = True
except ImportError:
    HAS_PROPHET = False
    print("Prophet not installed. Run: pip install prophet")

if HAS_PROPHET:
    # Prophet expects a dataframe with columns 'ds' (date) and 'y' (value)
    prophet_train = pd.DataFrame({"ds": y_train.index, "y": y_train.values})

    m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    m.fit(prophet_train)

    future = m.make_future_dataframe(periods=TEST_SIZE, freq="D")
    fc = m.predict(future)

    # Last TEST_SIZE rows of fc are the forecast
    prophet_forecast = pd.Series(fc["yhat"].values[-TEST_SIZE:], index=y_test.index)
    mae_prophet = mean_absolute_error(y_test, prophet_forecast)
    print(f"Prophet: Test MAE = £{mae_prophet:,.0f}")

    # Plot components
    m.plot_components(fc)
    plt.tight_layout()
    plt.show()
else:
    print("(Install prophet to run this section.)")

## Closing comparison

Across all methods we've now seen on the SAME 60-day holiday test window:

| Method | Approx MAE | Approach |
|---|---|---|
| Naive | £1,270 | Last-value (coincidentally good on this window) |
| Seasonal Naive | £2,600 | Lag-7 only |
| ETS-weekly | £2,050 | One-seasonality classical |
| ARIMA (best grid search) | varies | Univariate ARIMA, no seasonality |
| SARIMA(1,1,1)(1,1,1,7) | varies | One-seasonality classical (Bayesian fit) |
| ML-with-lags recursive | £2,550 | Multi-lag ML, errors compound over 60-day horizon |
| Prophet (if installed) | varies | Decomposition + flexible seasonality |

**Why the single-window numbers are unstable:**
- Test period includes the holiday spike (regime change)
- 60-step recursive forecasting compounds errors
- Models with only ~2 years of training data can't reliably learn annual cycles

**For deployment**, you want short-horizon (one-step or week-ahead) forecasts with regular retraining. On those, ML-with-lags and Prophet typically win cleanly.